### Harry potter rag 

In [1]:
!pip install "transformers<5.0.0" "numpy<2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 10.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 5.3 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.14.0
    Uninstalling huggingface_hub-1.14.0:
      Successfully uninstalled huggingface_hub-1.14.0
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [huggingface-hub]
    Found existing installation: transformers 5.8.132m0/2 [huggingface-hub]
    Uninstalling transformers-5.8.1:m╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-5.8.1━━━━━━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


In [2]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") # This is the API key for the Gemini API, which is used to access the language model.

csv_path = "HPrag.csv" 

df = pd.read_csv(csv_path)
df.head() 

,name,gender,job,house,patronus,species,blood_status,loyalty,skills,birth,...,titles,category,hand,light,difficulty,inventors,ingredients,side_effects,time,title
0,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,Phoenix,Human,Half-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Quirinus Quirrell,Male,Defence Against the Dark Arts(1991-1992),Ravenclaw,Non-corporeal,Human,Half-blood,Lord Voldemort,Learned in the theory of Defensive Magic| less...,"26 September,1970 or earlier",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fred Weasley,Male,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Beater,"1 April, 1978",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nymphadora Tonks,Female,Auror,Hufflepuff,"Jack rabbit, Wolf",Human,Half-blood,Ministry of Magic | Order of the Phoenix,"Talented Auror, Metamorphmagus",1 September 1972- 31 August 1973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lavender Brown,Female,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army |Hogwarts School of Witchcra...,Divination,1 September 1979- 31 August 1980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
import pandas as pd
from langchain_core.documents import Document

# 1. Läs in CSV-filen med Pandas (som vi vet fungerar!)
csv_path = "HPrag.csv"
df = pd.read_csv(csv_path)

# 2. Gör om varje rad i din dataframe till ett LangChain-dokument
documents = []
for index, row in df.iterrows():
    # Kombinera radens innehåll till en textsträng
    row_text = ", ".join([f"{col}: {val}" for col, val in row.items()])
    
    doc = Document(page_content=row_text, metadata={"row": index})
    documents.append(doc)

print(f"Antal laddade dokument: {len(documents)}")
print(f"Första dokumentet: {documents[0] if documents else 'Tomt'}")

Antal laddade dokument: 5766
Första dokumentet: page_content='name: Albus Percival Wulfric Brian Dumbledore, gender: Male, job: Headmaster, house: Gryffindor, patronus: Phoenix, species: Human, blood_status: Half-blood, loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry, skills: Considered by many to be one of the most powerful wizards of his time, birth: Late August 1881, death: 30 June, 1997 , source: characters, incantation: nan, type: nan, effect: nan, characteristics: nan, alias_names: nan, titles: nan, category: nan, hand: nan, light: nan, difficulty: nan, inventors: nan, ingredients: nan, side_effects: nan, time: nan, title: nan' metadata={'row': 0}


In [1]:
#Load the CSV file using the CSVLoader from langchain_community
from langchain_community.document_loaders.csv_loader import CSVLoader

csv_path = "HPrag.csv"
loader = CSVLoader(file_path=csv_path)
documents = loader.load()
print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0]}")

/Users/matildakosir/Documents/data_manager/datakvalitet/data_kvalit-_kunskapskontroll/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents: 5766
First document: page_content='name: Albus Percival Wulfric Brian Dumbledore
gender: Male
job: Headmaster
house: Gryffindor
patronus: Phoenix
species: Human
blood_status: Half-blood
loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Considered by many to be one of the most powerful wizards of his time
birth: Late August 1881
death: 30 June, 1997
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: ' metadata={'source': 'HPrag.csv', 'row': 0}


In [5]:
# Creating a embeddings model using the GoogleGenerativeAIEmbeddings class from langchain_google_genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    GEMINI_API_KEY=api_key
)


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/bh/166ml5hd70z9s6n98yg22k440000gn/T/ipykernel_92993/2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [7]:
# Create a Chroma vector store and add the documents to it
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="harry_potter_hf",
    embedding_function=embeddings,
    persist_directory="./chroma_harry_potter_hf_db",
)

In [8]:
# Add a smaller sample first before adding all documents, to make sure everything works as expected. 
# And to avoid hitting rate limits with the embedding model.
sample_documents = documents[:500]

document_ids = vector_store.add_documents(documents=sample_documents)

print(len(document_ids))

500


In [9]:
batch_size = 500

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]

    vector_store.add_documents(documents=batch)

    print(f"Added documents {i} to {i + len(batch)}")

Added documents 0 to 500
Added documents 500 to 1000
Added documents 1000 to 1500
Added documents 1500 to 2000
Added documents 2000 to 2500
Added documents 2500 to 3000
Added documents 3000 to 3500
Added documents 3500 to 4000
Added documents 4000 to 4500
Added documents 4500 to 5000
Added documents 5000 to 5500
Added documents 5500 to 5766


In [10]:
# Checking the similarity search with a sample question.
question = "What house does Harry Potter belong to?"

results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.page_content)
    print("---")

name: Henry Potter
gender: Male
job: ['Member of the Wizengamot (1913–1921)']
house: 
patronus: 
species: Human
blood_status: Pure-blood
loyalty: 
skills: 
birth: 
death: 
source: master
incantation: 
type: 
effect: 
characteristics: 
alias_names: ['Harry (by family and friends)']
titles: []
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: Kreacher
gender: Male
job: ["House of Black's house-elf (in/pre 1979–1996)", "Harry Potter's house-elf (1996–?)", 'Hogwarts kitchen worker (1996–?)']
house: 
patronus: 
species: House-elf
blood_status: 
loyalty: 
skills: 
birth: 
death: 
source: master
incantation: 
type: 
effect: 
characteristics: 
alias_names: []
titles: []
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: 
---
name: Harry James Potter
gender: Male
job: Student
house: Gryffindor
patronus: Stag
species: Human
blood_status: Half-blood
loyalty: Albus Dumbledore | Dumbledore's Army | Ord

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key
)

In [12]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
Answer the question like an old wise wizard, using the retrieved data as your source of information.
If the retrieved data does not contain the information needed to answer, say you don't know.

Question: {question}

Retrieved data:
{retrieved_data}
"""


In [14]:
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()

In [15]:
chain = (
    {
        "retrieved_data": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [22]:
question = input("Ask a question about the Harry Potter dataset: ")


In [23]:
answer = chain.invoke(question)

In [24]:
print(answer)

*Leans over a heavy, iron-bound tome and peers through half-moon spectacles.*

"Ah, seeker of secrets, you ask a question as vast as the Great Lake and as deep as the vaults of Gringotts! I have pored over these dusty scrolls—one for a Laxative Potion, another for the complex Wolfsbane, and a third for the restorative Healing Potion. They speak of many curious things: dragon blood, moonstone, bubotuber pus, and even the hair of a unicorn tail.

However, even with all my years wandering the corridors of ancient knowledge, the scrolls do not reveal the sum total of every substance in existence. I do not know how many ingredients exist in this wide, magical world."
